# Reproducing Gao et al. (2025) from dandiset 000463

**Paper.** Gao H, Ramachandran S, Yu K, He B (2025). Transcranial Focused Ultrasound Modulates
Feedforward and Feedback Cortico-Thalamo-Cortical Pathways by Selectively Activating Excitatory
Neurons. *J Neurosci* 45(23) e2218242025.
[doi:10.1523/jneurosci.2218-24.2025](https://doi.org/10.1523/jneurosci.2218-24.2025)

**Claims file.** `claims/000463/10.1523-jneurosci.2218-24.2025.yaml`, written from the version of
record as served by PubMed Central (PMC12139594), full text with Methods and legends; the paper has
no supplement and links no analysis code.

**Target dataset.** DANDI:000463, version `0.250829.1248`
([doi:10.48324/dandi.000463/0.250829.1248](https://doi.org/10.48324/dandi.000463/0.250829.1248)).
The paper cites no version of 000463 (it does not name 000463 at all), so the most recent published
version is pinned. All five versions, the draft included, list the same 29 files with the same
sizes (checked below), so the choice changes nothing.

**Where the paper's data are.** Dandiset 000463 cites the paper in its description, but holds
broadband recordings of one S1 probe from four rats (BH395, BH396, BH401, BH444, recorded November
2022 to February 2023), with no sorted units, no trial or stimulus times, and electrode locations
"unknown". None of its subjects is in the paper's own deposit. The paper says (Methods, "Data
acquisition and preprocessing"): "All experimental data has been made public in a repository at
the DANDI Archive (https://dandiarchive.org/dandiset/000946)." The claims are therefore marked
`data: elsewhere` and tested on DANDI:000946, version `0.250829.1249`
([doi:10.48324/dandi.000946/0.250829.1249](https://doi.org/10.48324/dandi.000946/0.250829.1249)),
the most recent published version; the paper cites no version of it, and its six versions list the
same files (checked below). Both dandisets are streamed; nothing is downloaded to disk.

**Claims.** 37 in the claims file. 2 are tested, 35 blocked.

| claim | status | reason |
|---|---|---|
| fig4-unit-counts | blocked | `metadata_insufficient`: no unit of 000946 is typed (all `celltype_label` 0) and no waveform is deposited |
| fig4a, fig4b, fig4-post-3000, fig4-30hz-responses, fig4-fsu-no-change | blocked | `metadata_insufficient`: RSU and FSU types needed |
| fig5a-sham-no-response | blocked | `data_missing`: no sham session in either dandiset |
| fig5b-control-cg1-no-response | blocked | `data_missing`: no CG1-targeted session (000946 has M1-targeted ones) |
| fig6a (2), fig6b, fig6c, fig6d, fig7a to fig7d, fig8a, fig8b (2), fig9a, fig9b | blocked | `metadata_insufficient`: RSU and FSU types needed |
| fig10a (3), fig10b, fig10c to fig10f | blocked | `metadata_insufficient`: pairs are by type |
| synthesis-prf-pathway | blocked | `metadata_insufficient`: depends on blocked claims |
| fig11-ac-unit-counts | tested | AC units are untyped in the paper |
| fig11b-ac-prf-responses | tested | AC units are untyped in the paper |
| fig11c-ac-s1-30hz-dsttc, fig11c-ac-s1fsu-30hz, fig11d-ac-s1-3000hz-dsttc | blocked | `metadata_insufficient`: S1 RSU or FSU types needed |
| methods-insitu-pressure | blocked | `data_missing`: no hydrophone scans in either dandiset |

In [ ]:
import time

import numpy as np
import pandas as pd
from scipy import stats

import tfus

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
T0 = time.time()

## Data check

### Dandiset 000463

Every file's metadata, read without the voltage data.

In [ ]:
meta463 = tfus.metadata(tfus.DANDISET, tfus.VERSION)
print(meta463["name"])
print("subjects (archive summary):", meta463["assetsSummary"]["numberOfSubjects"],
      "| variables:", meta463["assetsSummary"]["variableMeasured"])
a463 = tfus.assets(tfus.DANDISET, tfus.VERSION)
d463 = pd.DataFrame(tfus.pmap(tfus.describe_000463_file, a463))
assert "error" not in d463 or d463["error"].isna().all()
print(d463.drop(columns=["path", "device", "acquisition"]).to_string())
print()
print("files:", len(d463), "| subjects:", d463.subject.nunique(), sorted(d463.subject.unique()))
print("devices:", d463.device.unique(), "| acquisition series:", d463.acquisition.unique())
print("files with units:", int(d463.units.sum()), "| with any intervals table:", int((d463.intervals != "none").sum()),
      "| with stimulus series:", int((d463.stimulus != "none").sum()))
print("recording length, s: %.1f to %.1f; trials at 2.5 s per trial: %.0f to %.0f"
      % (d463.duration_s.min(), d463.duration_s.max(), d463.duration_s.min() / 2.5, d463.duration_s.max() / 2.5))

The 29 files hold 32-channel broadband voltage at 24414 Hz and a 3052 Hz LFP of one NeuroNexus
A1x32 probe each, from the four subjects the archive lists. There is no units table, no trials or
epochs table and no stimulus series, and every electrode's location is "unknown" (the electrode
group's is the placeholder "brain area"). Conditions appear only in the identifier (for example
`BH401_3000_200_67_5V`, PRF 3000 Hz, PD 200 us, UD 67 ms, 5 V, or `BH396_test5_24V`). The
description says recordings have 200 or 500 trials; the recording lengths give about 121 to 515
stimulation intervals of 2.5 s. The paper's analyses need units of S1 and POm (or AC), their types
and the sonication onsets, and the paper says 500 trials per condition and two probes per rat; none
of this can be read from 000463.

In [ ]:
rows = []
for ds in (tfus.DANDISET, tfus.PAPER_DANDISET):
    vs = tfus.versions(ds)
    pinned = tfus.VERSION if ds == tfus.DANDISET else tfus.PAPER_VERSION
    ref = {(a["path"], a["size"]) for a in tfus.assets(ds, pinned)}
    for v in vs:
        cur = {(a["path"], a["size"]) for a in tfus.assets(ds, v)}
        rows.append(dict(dandiset=ds, version=v, files=len(cur), differ_from_pinned=len(cur ^ ref)))
print(pd.DataFrame(rows).to_string(index=False))

Every version of each dandiset lists the same paths with the same sizes as the pinned one.

### Dandiset 000946, the paper's deposit

Every file's metadata, the units' spike counts and type labels, and the trial table, read without
the spike times (only 2000 spike times per file enter a signature used to find duplicates).

In [ ]:
meta946 = tfus.metadata(tfus.PAPER_DANDISET, tfus.PAPER_VERSION)
print(meta946["name"])
print(meta946["description"][:900])
a946 = tfus.assets(tfus.PAPER_DANDISET, tfus.PAPER_VERSION)
t = time.time()
s946 = pd.DataFrame(tfus.pmap(tfus.describe_000946_file, a946))
print("\nfiles read:", len(s946), "| failed:", int(s946["error"].notna().sum()) if "error" in s946 else 0,
      "| %.0f s" % (time.time() - t))
s946["kind"] = np.where(s946.spike_counts.notna(), "spikes", "STTC")
print(s946.groupby("kind").agg(files=("path", "size"), GB=("size", lambda x: x.sum() / 1e9)).to_string())
print("\nsubjects of 000463 found in 000946:", sorted(set(d463.subject) & set(s946.subject.dropna())))
print("000946 subjects:", s946.subject.nunique(), "| recording dates:", s946.session_start.min()[:10], "to",
      s946.session_start.max()[:10], "| 000463 dates:", d463.session_start.min()[:10], "to", d463.session_start.max()[:10])

In [ ]:
sp = s946[s946.kind == "spikes"].sort_values("path").copy()
# Recordings deposited more than once: same trial table, spike-count index and first and last spike times.
groups = sp.groupby("signature").path.apply(list)
print("spike files:", len(sp), "| distinct recordings:", len(groups),
      "| recordings by number of copies:", groups.apply(len).value_counts().sort_index().to_dict())
dup = sp[sp.duplicated("signature", keep=False)]
same_meta = dup.groupby("signature").agg(ids=("identifier", "nunique"), notes=("notes", "nunique"),
                                           areas=("area", "nunique"), starts=("session_start", "nunique"))
print("duplicate groups with one identifier:", int((same_meta.ids == 1).sum()), "of", len(same_meta),
      "| with one notes text:", int((same_meta.notes == 1).sum()),
      "| with one recorded area once the notes are parsed:", int((same_meta.areas == 1).sum()),
      "| with differing session_start_time:", int((same_meta.starts > 1).sum()))
for sig in same_meta.index[same_meta.areas > 1]:
    print("  same content, different areas:")
    print(dup[dup.signature == sig][["path", "identifier", "notes"]].to_string(index=False))
for paths in groups[groups.apply(len) > 1].head(3):
    print("   ", paths)
# Copies are sorted by path and the first is kept. A recording whose copies give different areas
# cannot be placed in either area, so all its files are left out.
conflict = same_meta.index[same_meta.areas > 1]
uniq = sp[~sp.signature.isin(conflict)].drop_duplicates("signature").copy()
print("\nrecordings left out because their copies give different areas:", len(conflict),
      sp[sp.signature.isin(conflict)].identifier.unique().tolist())
print("Each recording is counted once from here on:", len(uniq), "files.")

In [ ]:
labels = np.concatenate([l for l in uniq.labels if l is not None])
print("units:", int(uniq.spike_counts.apply(len).sum()), "| with a celltype_label column:", len(labels))
print("celltype_label values:", pd.Series(labels).value_counts().to_dict())
print("label descriptions:", uniq.label_description.value_counts().to_dict())
print("unit columns:", uniq.unit_columns.value_counts().to_dict())
print("\ntrials per recording:", uniq.n_trials.value_counts().head(8).to_dict())
print("trial stop minus start, s: %.3f to %.3f" % (uniq.trial_len_s.min(), uniq.trial_len_s.max()))
print("onset to onset, s: %.3f to %.3f (Methods: 2.5 s with 10%% jitter, 2.25 to 2.75)"
      % (uniq.isi_min.min(), uniq.isi_max.max()))
# The identifier's PRF and PD against the data_collection text of the same file.
dc = uniq.data_collection.str.extract(r"PRF:(\d+)Hz PD:([\d.]+)us UD:(\d+)ms")
ok = (dc[0].astype(float) == uniq.prf) & (dc[1].astype(float) == uniq.pd_us)
print("files whose data_collection gives PRF and PD:", int(dc[0].notna().sum()), "| agree with identifier:", int(ok.sum()))
print("files whose identifier does not parse:", uniq.identifier[uniq.prf.isna()].tolist())

The copies of a recording differ in their file names; in 27 groups the session start time also
differs by seconds, and in 27 the wording of `general/notes`. The group printed above holds one recording's spikes under both an
S1 and a POm file, so at least one of them is labelled with the wrong area; that recording is left
out from here on, since neither area can be trusted. The units table of
nearly every spike file has a `celltype_label` column described as "label 1 = RSU, label 2 = FSU",
but every one of its 22599 values is 0, and 33 recordings have no such column. No waveform, and
no IP or AHP measure, is deposited, so the RSU and FSU types of the paper (Methods, "Neuronal type
classification": k-means on each spike's initial phase and afterhyperpolarization period) cannot
be assigned. The PRF and PD of the identifier agree with the `data_collection` text in every file that has
both (the one other file, `BH594_coating_3000_200_10`, has an identifier that does not parse). Trials last 2.2 s in the table and onsets are 2.25 to 2.75 s apart, as the Methods
say. Most recordings have 500 trials, as the description says; some have 400 to 499.

In [ ]:
recs = uniq.copy()
recs["group"] = np.where(recs.subject.isin(recs[recs.area == "ac"].subject), "AC group",
                         np.where(recs.subject.isin(recs[recs.area == "pom"].subject), "POm group", "other"))
print("recordings by area recorded and target stimulated (from general/notes):")
print(pd.crosstab(recs.area, recs.target).to_string())
print("\nsubjects by group:", recs.groupby("group").subject.unique().apply(lambda s: sorted(x for x in s if isinstance(x, str))).to_dict())
print("\nidentifier suffixes:", recs.suffix.value_counts().to_dict())
print("\nrecordings targeted at S1 with plain identifiers, by PRF and PD (us), per area:")
plain = recs[(recs.target == "s1") & recs.suffix.isin(["", "_POM_S1"])]
print(pd.crosstab([plain.prf, plain.pd_us], plain.area).to_string())
print("\nvoltage field of the identifier (V), all recordings:", recs.volt.value_counts(dropna=False).to_dict())
print("files whose notes give no recorded area:")
print(recs[~recs.area.isin(["s1", "pom", "ac", "m1"])][["path", "identifier", "notes"]].to_string(index=False))


The files record one area each; the two probes of one recording are two files with the same
identifier and trial table. Areas recorded are S1, POm, AC, M1 and a few others, and targets are
S1, M1 and POm (identifier suffixes `_MOTOR`, `_POM_MOTOR` and `_POM_POM`). There is no session
without ultrasound (sham, Figure 5A) and none targeted at the cingulate cortex CG1 (control, Figure
5B); M1-targeted sessions exist, which the paper does not describe. The PRF and PD pairs cover the
paper's constant-PD series (PD 200 us), its DC series at 3000 Hz (PD 2, 20, 100, 200, 300 us) and at
30 Hz (PD 200, 2000, 10000, 20000, 30000 us), and its 60% DC series, and add series at 1500, 300 and
4500 Hz that the paper does not report. The AC group has 7 subjects (paper: 12) and the subjects
with POm recordings number 10 (paper: 16). The voltage field of the identifier is 5 V in every
recording whose identifier parses, so none is marked as a session without ultrasound. The 14 files
whose notes name no main area are BH494's `M1_r` and `S1_r`, BH499's S1BF and VPM recordings, five
BH500 files whose notes leave the area empty, and BH538's `S1_1`, `AC_1` and `AC_2` (PD 40 us);
their stimulation target is S1 or M1, never another site.

### Clock of the spikes and the trial table

The AC recordings of the constant-PD series (the data of the two tested claims) and the S1
recordings of the same sessions are loaded. The two files of one session must have the same
trial table, and the population rate must respond at the trial onsets and not at times shifted
into the middle of the intersonication interval.

In [ ]:
ac_files = recs[(recs.area == "ac") & (recs.target == "s1") & (recs.pd_us == 200) & (recs.suffix == "")]
s1_mates = recs[(recs.area == "s1") & recs.identifier.isin(ac_files.identifier)]
t = time.time()
loaded = {d["path"]: d for d in tfus.pmap(tfus.load_spikes, [dict(path=p, asset_id=a) for p, a in
          zip(pd.concat([ac_files, s1_mates]).path, pd.concat([ac_files, s1_mates]).asset_id)])}
print("files loaded:", len(loaded), "| failed:", sum("error" in d for d in loaded.values()), "| %.0f s" % (time.time() - t))
rows = []
for _, a in ac_files.iterrows():
    mate = s1_mates[s1_mates.identifier == a.identifier]
    same = [np.array_equal(loaded[a.path]["onsets"], loaded[m]["onsets"]) for m in mate.path]
    rows.append(dict(identifier=a.identifier, trials=a.n_trials, s1_files=len(mate), same_trial_table=all(same) and len(same) > 0))
print(pd.DataFrame(rows).to_string(index=False))
# window_counts uses searchsorted, which needs each unit's spike times in increasing order.
unsorted = [(p, k) for p, d in loaded.items() for k, u in enumerate(d["units"]) if np.any(np.diff(u) < 0)]
print("units whose spike times are not in increasing order:", len(unsorted), "of",
      sum(len(d["units"]) for d in loaded.values()))


In [ ]:
def population_psth(files, shift=0.0, edges=np.arange(-0.2, 0.5001, 0.01)):
    total = np.zeros(len(edges) - 1)
    n_trials = 0
    for p in files:
        d = loaded[p]
        on = d["onsets"] + shift
        for u in d["units"]:
            if len(u) > tfus.MIN_SPIKES:
                rel = np.concatenate([u[(u >= o + edges[0]) & (u < o + edges[-1])] - o for o in on])
                total += np.histogram(rel, edges)[0]
        n_trials += len(on)
    return edges, total / total[edges[:-1] < 0].mean()

for area, files in (("AC", ac_files.path), ("S1", s1_mates.path)):
    for shift in (0.0, 1.0):
        e, r = population_psth(files, shift)
        during = r[(e[:-1] >= 0) & (e[:-1] < 0.067)].mean()
        post = r[(e[:-1] >= 0.201) & (e[:-1] < 0.268)].mean()
        print(f"{area}, onsets shifted by {shift:.1f} s: rate relative to -200 to 0 ms, 0 to 67 ms {during:.3f}, "
              f"201 to 268 ms {post:.3f}, peak {r.max():.2f} at {e[np.argmax(r)] * 1000:.0f} ms")
    e, r = population_psth(files, 0.0, edges=np.arange(-0.1, 0.1401, 0.02))
    print(f"   {area}, 20 ms bins from -100 ms:", " ".join(f"{x * 1000:.0f}:{y:.2f}" for x, y in zip(e[:-1], r)))

Of the 17 AC recordings of the series, 16 have an S1 file with the same identifier, and in each the trial table is identical; BH537_1500_200_67_5 has no S1 file. At the true onsets
the pooled AC and S1 rates rise after onset (numbers above), and with the onsets moved 1 s later,
into the intersonication interval, they stay near 1. In 20 ms bins the S1 rate steps up in the
bin that starts at 0 ms and falls back after 60 ms, the length of the 67 ms sonication. So the trial `start_time` is consistent with the
sonication onset on the spike clock; that it is the onset to the millisecond is not shown. No unit of the loaded files has spike times
out of increasing order (printed above), which `window_counts` needs.

## Blocked claims

Every claim on S1 or POm units uses the RSU and FSU types, which 000946 does not give (all
`celltype_label` values are 0) and which cannot be computed without waveforms. The counts below show
what the dataset holds for the unit counts of Figures 4 and 6, untyped. They decide no outcome,
but they bound it: the RSU and FSU counts of a region and condition add up to the untyped total,
so both can be within 10% only if the total is within the sum of the two ranges. That fails in all
six region and condition groups, so at most 6 of the 12 counts could match under any typing:
fig4-unit-counts cannot be reproduced, and whether it is partial (6 to 9) or not reproduced
depends on the types.

In [ ]:
def count_units(df, threshold=tfus.MIN_SPIKES):
    return df.spike_counts.apply(lambda c: int((c > threshold).sum()))

pom_subjects = recs[(recs.area == "pom") & (recs.target == "s1")].subject.unique()
main = recs[(recs.target == "s1") & recs.subject.isin(pom_subjects) & recs.area.isin(["s1", "pom"])
            & recs.suffix.isin(["", "_POM_S1", "_POM _S1"])].copy()
main["units"] = count_units(main)
cp = main[main.pd_us == 200]
tab = cp.pivot_table(index="prf", columns="area", values="units", aggfunc="sum")
tab["sessions"] = cp[cp.area == "s1"].groupby("prf").size()
print("Units with more than 2000 spikes, untyped, in the constant-PD series of the subjects with POm recordings:")
print(tab.to_string())
print("Paper (RSU + FSU): 3000 Hz S1 312 + 54 = 366, POm 233 + 17 = 250; 30 Hz S1 148 + 61 = 209, POm 189 + 43 = 232;"
      " all five PRFs S1 1050 + 247 = 1297, POm 913 + 106 = 1019.")
print("Totals over five PRFs here: S1", int(tab["s1"].sum()), "POm", int(tab["pom"].sum()))
print("\nSessions and units of the constant-PD series by subject, PRF and area:")
print(cp.pivot_table(index=["subject", "prf"], columns="area", values="units", aggfunc=["size", "sum"]).fillna(0).astype(int).to_string())

# The criterion of fig4-unit-counts compares 12 counts by type. Whatever the typing, RSU + FSU equals
# the untyped total, so both counts of a region and condition can be within 10% only if the total is
# within the sum of the two tolerance ranges.
paper = {("s1", 3000): (312, 54), ("pom", 3000): (233, 17), ("s1", 30): (148, 61), ("pom", 30): (189, 43),
         ("s1", "all"): (1050, 247), ("pom", "all"): (913, 106)}
rows = []
for (area, prf), (r, f) in paper.items():
    n = int(tab[area].sum()) if prf == "all" else int(tab.loc[prf, area])
    lo, hi = 0.9 * (r + f), 1.1 * (r + f)
    rows.append(dict(area=area, prf=prf, untyped=n, paper_rsu=r, paper_fsu=f, both_possible_range=f"{lo:.0f} to {hi:.0f}",
                     max_within=2 if lo <= n <= hi else 1))
b = pd.DataFrame(rows)
print(b.to_string(index=False))
print("most counts that any typing could put within tolerance:", int(b.max_within.sum()), "of 12 (reproduced needs 10)")


### The precomputed STTC files

000946 also holds STTC files. Their notes are listed, and one is opened to list what its units table
holds, to see whether pairs carry a cell type.

In [ ]:
st = s946[s946.kind == "STTC"].sort_values("path")
# The areas each STTC file names, from general/notes, without opening the arrays.
print(st.notes.str.lower().str.replace(" ", "").value_counts().to_string())
print("STTC files:", len(st), "| shapes of the undirected STTC array, first 5:", st.sttc_shape.head().tolist())
d = tfus.describe_sttc_file(dict(path=st.path.iloc[0], asset_id=st.asset_id.iloc[0]))
print(d["path"], "|", d["identifier"], "|", d["notes"])
print("units description:", d["units_description"][:700])
for k, (shape, desc) in d["datasets"].items():
    print(f"  {k}: shape {shape}; {desc}")


### Untyped pairs

Untyped pairs for fig11c and fig11d. The claims pair AC neurons with S1 RSUs or FSUs; untyped,
the AC units and S1 units with more than 2000 spikes recorded in the same constant-PD session give
an upper bound on the pairs of each type. They decide nothing.

The notes of 364 of the 370 STTC files name AC, POm or M1 against S1; the other six name S1BF, VPM,
two AC probes or no area. The one file opened (M1 against S1) holds 4-D arrays, such as (10, 499, 20, 9),
whose description lists five dimensions (lag dt, the two units, trial and the ten 67 ms windows from
-67 to 603 ms), so the order of the dimensions is not established. Its units table has no cell type,
so it does not supply the types either; the other files were not opened. The untyped pairs are
15430 over the constant-PD series, 3760 at 30 Hz and 4303 at 3000 Hz, each above the paper's
numbers (1421 and 878 AC-S1 RSU pairs, 1714 pairs at 3000 Hz), as an upper bound must be.

In [ ]:
ac_cp = recs[(recs.area == "ac") & (recs.target == "s1") & (recs.pd_us == 200) & (recs.suffix == "")].copy()
s1_cp = recs[(recs.area == "s1") & recs.identifier.isin(ac_cp.identifier)].copy()
ac_cp["n_ac"], s1_cp["n_s1"] = count_units(ac_cp), count_units(s1_cp)
pr = ac_cp[["identifier", "subject", "prf", "n_ac"]].merge(s1_cp[["identifier", "n_s1"]], on="identifier", how="left")
pr["pairs"] = pr.n_ac * pr.n_s1.fillna(0)
print(pr.to_string(index=False))
print("\nAC x S1 pairs, untyped, by PRF:", pr.groupby("prf").pairs.sum().astype(int).to_dict(), "| all:", int(pr.pairs.sum()))
print("Paper: 1421 AC-S1 RSU pairs, of which 878 at 30 Hz; 1714 pairs at 3000 Hz, of which 1016 AC-S1 FSU.")


## fig11-ac-unit-counts

> The auditory-confound analysis rests on 453 AC neurons over the PRFs, with 83, 107, 104 and 100
> neurons in the 30, 1500, 3000 and 4500 Hz sessions.

AC units with more than 2000 spikes in the session (M1), counted once per session, per PRF. The
criterion names two readings of the sessions: the constant-PD series (PD 200 us, which the Methods
use for "the PRF effect exploration") and every AC session the dataset labels with these PRFs.

In [ ]:
acr = recs[(recs.area == "ac") & (recs.target == "s1") & (recs.suffix == "")].copy()
acr["units"] = count_units(acr)
acr["all_units"] = acr.spike_counts.apply(len)
expected = {"all": 453, 30: 83, 1500: 107, 3000: 104, 4500: 100}

def count_table(df):
    by = df.groupby("prf").agg(sessions=("path", "size"), subjects=("subject", "nunique"),
                               units=("units", "sum"), units_no_threshold=("all_units", "sum"))
    by.loc["all"] = by.sum()
    return by

for name, df in (("(a) constant-PD series, PD 200 us", acr[acr.pd_us == 200]),
                 ("(b) every AC session with these PRFs, all PDs", acr)):
    tab = count_table(df)
    tab["paper"] = [expected.get(k, np.nan) for k in tab.index]
    tab["within_10pct"] = [abs(u - e) <= 0.1 * e if e == e else np.nan for u, e in zip(tab.units, tab.paper)]
    print(name); print(tab.to_string())
    print("counts within tolerance:", int(np.nansum(tab.within_10pct.astype(float))), "of 5\n")
print("Constant-PD series by session:")
print(acr[acr.pd_us == 200][["identifier", "n_trials", "all_units", "units"]].to_string(index=False))

**Result: not reproduced.** In the constant-PD series the dandiset has 112, 138, 167 and 127 AC units
with more than 2000 spikes at 30, 1500, 3000 and 4500 Hz (633 with 89 at 300 Hz), against 83, 107,
104, 100 and 453; none of the five counts is within 10%. Taking every AC session with these PRFs
gives larger counts still, also none within tolerance. The series comes from 6 of the 7 AC-group
subjects, 3 or 4 sessions per PRF.

## fig11b-ac-prf-responses

> AC neurons show time-locked responses to 1500 and 3000 Hz PRF tFUS targeting S1 and
> postsonication responses to 30, 1500, 3000 and 4500 Hz, with the least response to 300 Hz.

AC units of the constant-PD series (selected as above), normalized rate (M3) in 0 to 67 ms and 201
to 268 ms. Kruskal-Wallis across the five PRFs at p < 0.01, then each PRF against baseline,
one-tailed greater, Bonferroni-corrected p < 0.01 (M4). Every reading of O2 (normalization), O3
(baseline sample) and O4 (Bonferroni factor: 2 windows, or 5 PRFs) is run. The constant-PD series is
taken from the Methods ("For the PRF effect exploration, the pulse duration (PD) was set at 200 μs
for all the PRFs"); every AC session at these PRFs is shown as a variant.

In [ ]:
PRFS = [30, 300, 1500, 3000, 4500]

def unit_rates(files):
    rows = []
    for _, f in files.iterrows():
        d = loaded[f.path]
        for k, u in enumerate(d["units"]):
            if len(u) <= tfus.MIN_SPIKES:
                continue
            r = dict(subject=f.subject, identifier=f.identifier, prf=int(f.prf), unit=k)
            for o2 in "ab":
                r[f"pre2_{o2}"] = tfus.normalized_rate(u, d["onsets"], tfus.WIN_PRE2, o2)
                r[f"during_{o2}"] = tfus.normalized_rate(u, d["onsets"], tfus.WIN_DURING, o2)
                r[f"post_{o2}"] = tfus.normalized_rate(u, d["onsets"], tfus.WIN_POST, o2)
            rows.append(r)
    return pd.DataFrame(rows)

def judge(R):
    out, detail = [], []
    for o2 in "ab":
        kw = {w: stats.kruskal(*[R[R.prf == p][f"{w}_{o2}"].dropna() for p in PRFS]).pvalue for w in ("during", "post")}
        means = {w: {p: R[R.prf == p][f"{w}_{o2}"].mean() for p in PRFS} for w in ("during", "post")}
        for o3 in "abc":
            raw = {}
            for w in ("during", "post"):
                for p in PRFS:
                    x = R[R.prf == p][[f"{w}_{o2}", f"pre2_{o2}"]].dropna()
                    raw[(w, p)] = tfus.baseline_test(x[f"{w}_{o2}"], x[f"pre2_{o2}"], o3)
            for o4, k in (("a", 2), ("b", 5)):
                sig = {key: tfus.bonferroni(v, k) < tfus.ALPHA for key, v in raw.items()}
                named = [sig[("during", 1500)], sig[("during", 3000)], sig[("post", 30)], sig[("post", 1500)],
                         sig[("post", 3000)], sig[("post", 4500)]]
                excluded_ok = not any(sig[("during", p)] for p in (30, 300, 4500)) and not sig[("post", 300)]
                lowest = all(min(means[w], key=means[w].get) == 300 for w in ("during", "post"))
                kw_ok = kw["during"] < tfus.ALPHA and kw["post"] < tfus.ALPHA
                if kw_ok and all(named) and excluded_ok and lowest:
                    outcome = "reproduced"
                elif kw_ok and sum(named) >= 4:
                    outcome = "partially_reproduced"
                else:
                    outcome = "not_reproduced"
                out.append(dict(O2=o2, O3=o3, O4=f"{o4} (x{k})", KW_during=kw["during"], KW_post=kw["post"],
                                named_held=sum(named), exclusions_hold=excluded_ok, lowest_300=lowest, outcome=outcome,
                                significant_during=",".join(str(p) for p in PRFS if sig[("during", p)]),
                                significant_post=",".join(str(p) for p in PRFS if sig[("post", p)])))
                for key, v in raw.items():
                    detail.append(dict(O2=o2, O3=o3, O4=o4, window=key[0], prf=key[1], p_raw=v,
                                       p_corrected=tfus.bonferroni(v, k)))
    return pd.DataFrame(out), pd.DataFrame(detail)

R = unit_rates(ac_files)
print("units:", len(R), "| by PRF:", R.groupby("prf").size().to_dict(),
      "| without a prestimulation spike (dropped, O2a):", int(R.during_a.isna().sum()))
summary = R.groupby("prf")[["during_a", "post_a", "during_b", "post_b", "pre2_a", "pre2_b"]].agg(["mean", "sem", "median"])
print(summary.round(1).to_string())
print("ranges, O2a: during %.1f to %.1f, post %.1f to %.1f" % (R.during_a.min(), R.during_a.max(), R.post_a.min(), R.post_a.max()))
res, det = judge(R)
print(res.to_string(index=False, float_format=lambda v: "%.3g" % v))

In [ ]:
print("Raw and corrected p of each PRF against baseline, reading O2a (one-tailed greater):")
piv = det[det.O2 == "a"].pivot_table(index=["window", "prf"], columns=["O3", "O4"], values="p_corrected")
print(piv.to_string(float_format=lambda v: "%.2g" % v))
print("\nBy subject (one session per subject and PRF): mean normalized rate, O2a, and units")
by = R.groupby(["prf", "subject"]).agg(units=("unit", "size"), during=("during_a", "mean"), post=("post_a", "mean"))
print(by.round(1).to_string())
print("\nSessions whose mean is above 100%:",
      {w: f"{int((by[w] > 100).sum())} of {len(by)}" for w in ("during", "post")})

Two checks on the readings. First, O2a and O2b on synthetic units with no stimulus effect:
Poisson spike trains at fixed rates, 500 trials 2.5 s apart. A normalization on the paper's scale,
where 100% is no change, should give about 100% at every rate. Second, how much the result rests
on single subjects: the judgement repeated with each subject left out.

In [ ]:
rng = np.random.default_rng(46311)
onsets_syn = np.arange(1.0, 1.0 + 500 * 2.5, 2.5)
rows = []
for rate in (2, 5, 15, 40):
    vals = {"a": [], "b": []}
    for _ in range(200):
        n = rng.poisson(rate * (onsets_syn[-1] + 2.0))
        u = np.sort(rng.uniform(0, onsets_syn[-1] + 2.0, n))
        for o2 in "ab":
            vals[o2].append(tfus.normalized_rate(u, onsets_syn, tfus.WIN_DURING, o2))
    rows.append(dict(rate_hz=rate, O2a_mean=np.nanmean(vals["a"]), O2a_sd=np.nanstd(vals["a"]),
                     O2b_mean=np.nanmean(vals["b"]), O2b_sd=np.nanstd(vals["b"])))
print("Synthetic units without a stimulus effect, normalized rate in 0 to 67 ms (200 units per rate):")
print(pd.DataFrame(rows).round(1).to_string(index=False))
print("\nAC units' prestimulation rate, Hz: median %.1f, range %.1f to %.1f" % tuple(
    np.percentile([tfus.window_counts(u, loaded[f.path]["onsets"], tfus.WIN_PRE).mean() / 0.067
                   for _, f in ac_files.iterrows() for u in loaded[f.path]["units"] if len(u) > tfus.MIN_SPIKES], [50, 0, 100])))

print("\nWith each subject left out (O2a; outcome under each of the six O3 x O4 readings):")
loso = []
for s in sorted(R.subject.unique()):
    r_s, _ = judge(R[R.subject != s])
    r_s = r_s[r_s.O2 == "a"]
    loso.append(dict(left_out=s, units=int((R.subject != s).sum()), outcomes=r_s.outcome.value_counts().to_dict(),
                     named_held=f"{r_s.named_held.min()} to {r_s.named_held.max()}",
                     lowest_300=int(r_s.lowest_300.sum()), KW_during_max=r_s.KW_during.max(),
                     significant_during=sorted(set(r_s.significant_during))))
print(pd.DataFrame(loso).to_string(index=False, float_format=lambda v: "%.2g" % v))


In [ ]:
# Variant: every AC session at these PRFs, all PDs (the second reading of the session set in
# fig11-ac-unit-counts). Loads the remaining AC files.
rest = acr[acr.prf.isin(PRFS) & ~acr.path.isin(list(loaded))]
t = time.time()
loaded.update({d["path"]: d for d in tfus.pmap(tfus.load_spikes, [dict(path=p, asset_id=a) for p, a in zip(rest.path, rest.asset_id)])})
print("files loaded:", len(rest), "| %.0f s" % (time.time() - t))
print("units whose spike times are not in increasing order, all loaded files:",
      sum(np.any(np.diff(u) < 0) for d in loaded.values() for u in d["units"]))
R_all = unit_rates(acr[acr.prf.isin(PRFS)])
print("units:", len(R_all), "| by PRF:", R_all.groupby("prf").size().to_dict())
print(R_all.groupby("prf")[["during_a", "post_a"]].mean().round(1).to_string())
res_all, _ = judge(R_all)
print(res_all.to_string(index=False, float_format=lambda v: "%.3g" % v))

**Result: partially reproduced, resting on the assumption of the O2a normalization.** Under O2a (mean
window rate over mean prestimulation rate) both Kruskal-Wallis tests are significant, the six named
results hold (1500 and 3000 Hz during sonication; 30, 1500, 3000 and 4500 Hz after it), 300 Hz has the
lowest mean in both windows, and 300 Hz is not significant after sonication; but 4500 Hz is also
significant during sonication under every O3 and O4 reading, 30 Hz under three of the six and
300 Hz under the two O3b readings, where the claim says only 1500 and 3000 Hz respond. Under O2b (mean of per-trial ratios, trials without a prestimulation spike left out) the
means of every PRF and window, the 134 to 67 ms baseline window included, are 54 to 74%, the
during-sonication Kruskal-Wallis test is not significant (p = 0.021), and at most 2 of the 6 named
results hold; the claim is not reproduced under it. O2a is adopted: under O2b a window with an
unchanged rate does not come out near 100%, while the paper reports its normalized rates on a scale
where 100% is no change (sham and control p = 1.0, responses reported at 123 to 159% in the Results text). The mean responses
differ much between sessions (by subject, above), but leaving out any one subject gives partial or
full reproduction under every O2a reading, never not reproduced. With every AC session at these
PRFs (all PDs) the outcome under O2a is also partial, and there 300 Hz is not the lowest.

The synthetic units support the choice of O2a: without any stimulus effect O2a gives 99.9 to 101.6% at
2 to 40 Hz, while O2b gives 13% at 2 Hz, 31% at 5 Hz, 77% at 15 Hz and 127% at 40 Hz, so O2b
depends on the firing rate (the AC units fire at a median 6 Hz before onset) and does not put no
change at 100%. The claims file's header blocks a claim whose method readings disagree; here the
two O2 readings disagree, but O2b is not a normalization on the paper's scale, so the outcome rests
on O2a as an assumption and every reading is shown.

## Summary

| claim | outcome |
|---|---|
| fig11-ac-unit-counts | not reproduced: 0 of 5 counts within 10% (dataset has more AC units) |
| fig11b-ac-prf-responses | partially reproduced (assumed O2a): named responses hold, but 4500 Hz also responds during sonication |
| 31 claims on S1 and POm units by type (fig4 to fig10, fig11c, fig11d) and synthesis-prf-pathway | blocked, `metadata_insufficient` |
| fig5a-sham-no-response, fig5b-control-cg1-no-response, methods-insitu-pressure | blocked, `data_missing` |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The data are from 000946, the paper's deposit, not the
target 000463. The AC sessions are those of the dataset's constant-PD series (6 subjects; the paper
reports 12 rats in the AC group). The normalization of the rate, the baseline sample and the
Bonferroni factor are not fixed by the paper; every reading is shown.

What was found about the dataset. Dandiset 000463 cites the paper but holds none of its recordings:
broadband voltage of one S1 probe from four other rats, with no units, no trial or stimulus times
and no electrode locations. The paper's deposit 000946 holds sorted units and trial tables, but every
unit's `celltype_label` is 0 although the column says 1 is RSU and 2 is FSU, and no waveforms are
deposited, so the paper's RSU and FSU types cannot be recovered; most spike files are deposited
twice under different file names; there are no sham or CG1-targeted
sessions; and it holds more AC units than the paper reports.

In [ ]:
print("notebook runtime: %.1f min" % ((time.time() - T0) / 60))